# geo0266 – Least squares and the SVD: a line fit step by step

**Unit:** geo0260 GNSS, RTK and SAPOS · **Builds on:** geo0264 §1.2–§1.4 (*The maths*, the solver, PDOP) · **Data:** none (five invented points) · **Optional:** self-study for everyone who wants to see what `np.linalg.lstsq` does inside

In geo0264 the GNSS solver calls `np.linalg.lstsq` in every iteration. This notebook opens that black box with the smallest example that still shows everything: a straight line through five points. You set up the least-squares problem, compute the **singular value decomposition (SVD)** of the design matrix by hand, solve the problem with it and then read from the singular values how well the unknowns are determined, which is exactly what PDOP does for GNSS.

You will learn to
1. set up an overdetermined linear system and its least-squares solution (residuals, normal equations),
2. compute the SVD $A = U\Sigma V^T$ by hand via $A^TA$ and check it against NumPy,
3. solve least squares with the SVD and see where the residuals live,
4. read conditioning and rank from the singular values,
5. connect it back to GNSS: covariance and PDOP from the SVD.

Levels: §1–§4 core (about 1.5 h), §5–§7 stretch. Every exercise cell is followed by a check, so you know whether you are right before you go on.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1. The problem: a line through five points

We fit a straight line $y = a + b\,x$ through five points that lie roughly on $y = 1 + 2x$:

| $x$ | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| $y$ | 1.1 | 2.9 | 5.2 | 6.8 | 9.1 |

Each point gives one equation $a + b\,x_i = y_i$. Written as a matrix equation $A\,\mathbf{p} \approx \mathbf{y}$:

$$\underbrace{\begin{bmatrix}1&0\\1&1\\1&2\\1&3\\1&4\end{bmatrix}}_{A\ (5\times 2)} \underbrace{\begin{bmatrix}a\\b\end{bmatrix}}_{\mathbf{p}} \approx \underbrace{\begin{bmatrix}1.1\\2.9\\5.2\\6.8\\9.1\end{bmatrix}}_{\mathbf{y}}$$

Five equations, two unknowns: the system is **overdetermined**. The points do not lie exactly on a line, so there is no exact solution, only a best compromise.

This is the structure of the GNSS solver in geo0264: one row per observation (satellite), one column per unknown (X, Y, Z, clock). The line fit is linear, so we need no iteration; one least-squares step is the answer.

In [ ]:
x = np.array([0, 1, 2, 3, 4.0])
y = np.array([1.1, 2.9, 5.2, 6.8, 9.1])
A = np.c_[np.ones_like(x), x]          # column of ones (for a), column of x (for b)
print(A)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(x, y, "o", color="tab:blue")
ax.set(xlabel="x", ylabel="y", title="Five points")
ax.grid(alpha=0.3)
plt.show()

### 1.1 Residuals and least squares
For any parameters $\mathbf{p} = (a, b)$ the **residuals** are what the line fails to explain:

$$\mathbf{v} = \mathbf{y} - A\,\mathbf{p}$$

Least squares chooses the $\mathbf{p}$ that makes the **sum of squared residuals** smallest:

$$S(\mathbf{p}) = \lVert \mathbf{y} - A\,\mathbf{p} \rVert^2 = \sum_i v_i^2 \;\rightarrow\; \min$$

At the minimum the gradient is zero: $\partial S / \partial \mathbf{p} = -2A^T(\mathbf{y} - A\,\mathbf{p}) = \mathbf{0}$. This gives the **normal equations**

$$A^TA\,\mathbf{p} = A^T\mathbf{y}$$

*Link to geo0264:* there the right-hand side was $\boldsymbol{\ell}$ (observed minus computed at the current guess) and the unknowns were the corrections $\Delta\mathbf{p}$. For a linear problem started at $\mathbf{p}_0 = \mathbf{0}$, $\boldsymbol{\ell} = \mathbf{y}$ and one step gives the final answer.

**Exercise:** first *predict* the four entries of $A^TA$ from the $x$ values (hint: number of points, $\sum x_i$, $\sum x_i^2$). Then compute `N` $= A^TA$, `n` $= A^T\mathbf{y}$ and solve the normal equations with `np.linalg.solve`.

In [ ]:
# YOUR CODE HERE
print("A^T A =\n", N)
print("A^T y =", n)
print("a, b  =", p_normal)
assert np.allclose(p_normal, [1.04, 1.99]), "expected a = 1.04, b = 1.99"

In [ ]:
a, b = p_normal
v = y - A @ p_normal                   # residuals
print("residuals v =", v.round(3), "  sum of squares S =", round(float(v @ v), 3))

xx = np.linspace(-0.3, 4.3, 50)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(xx, a + b * xx, color="tab:orange", label=f"y = {a:.2f} + {b:.2f} x")
ax.vlines(x, a + b * x, y, color="tab:red", lw=2, label="residuals")
ax.plot(x, y, "o", color="tab:blue")
ax.set(xlabel="x", ylabel="y", title="Least-squares line")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 2. What the SVD is: the idea first

The SVD answers a geometric question: **what does the matrix $A$ do to the parameter vectors** $\mathbf{p} = (a, b)$?

$A$ maps every $\mathbf{p}$ to $A\,\mathbf{p}$: the five $y$ values of the line with those parameters. Take all $\mathbf{p}$ of length 1, the unit circle in the $(a, b)$ plane. Their images $A\,\mathbf{p}$ form an **ellipse** in the five-dimensional measurement space (a flat one, lying in the plane spanned by the two columns of $A$). The SVD finds

* $V$: the two directions $\mathbf{v}_1, \mathbf{v}_2$ in parameter space that are mapped onto the **axes** of the ellipse,
* $\Sigma$: the lengths of the semi-axes, the **singular values** $s_1 \ge s_2 \ge 0$,
* $U$: the directions $\mathbf{u}_1, \mathbf{u}_2$ of these axes in measurement space.

In one line, for every $i$:

$$A\,\mathbf{v}_i = s_i\,\mathbf{u}_i$$

Collecting the columns gives $A V = U \Sigma$, and because $V$ is orthogonal ($V^{-1} = V^T$):

$$A = U\,\Sigma\,V^T \qquad \underbrace{A}_{5\times 2} = \underbrace{U}_{5\times 2}\;\underbrace{\Sigma}_{2\times 2}\;\underbrace{V^T}_{2\times 2}$$

$U$ and $V$ have orthonormal columns ($U^TU = I$, $V^TV = I$); $\Sigma$ is diagonal. Read from right to left, $A$ does three simple things: **rotate** ($V^T$), **stretch** along the axes ($\Sigma$), **place** the result in measurement space ($U$).

A five-dimensional ellipse cannot be drawn, but its plane can: below we show it in the coordinates $(\mathbf{u}_1, \mathbf{u}_2)$. NumPy's `svd` is used here only for the picture; in §3 you compute everything yourself.

In [ ]:
U_np, s_np, Vt_np = np.linalg.svd(A, full_matrices=False)

t = np.linspace(0, 2 * np.pi, 200)
circle = np.vstack([np.cos(t), np.sin(t)])          # unit vectors p in the (a, b) plane
ellipse = U_np.T @ (A @ circle)                     # A p, expressed in the (u1, u2) plane

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
ax1.plot(*circle, color="grey")
for i, c in enumerate(["tab:blue", "tab:orange"]):
    ax1.annotate("", Vt_np[i], (0, 0), arrowprops=dict(arrowstyle="->", color=c, lw=2))
    ax1.text(*(1.12 * Vt_np[i]), f"v{i + 1}", color=c, fontsize=12)
ax1.set(xlabel="a", ylabel="b", title="parameter space: unit circle", aspect="equal")
ax2.plot(*ellipse, color="grey")
for i, c in enumerate(["tab:blue", "tab:orange"]):
    tip = np.eye(2)[i] * s_np[i]
    ax2.annotate("", tip, (0, 0), arrowprops=dict(arrowstyle="->", color=c, lw=2))
    ax2.text(*(tip + [0.15, 0.15]), f"s{i + 1} u{i + 1}", color=c, fontsize=12)
ax2.set(xlabel="u1", ylabel="u2", title="measurement space: ellipse A p", aspect="equal",
        xlim=(-6.5, 7.5), ylim=(-2.5, 2.5))
for ax in (ax1, ax2):
    ax.grid(alpha=0.3); ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
plt.tight_layout(); plt.show()
print("singular values:", s_np)

## 3. The SVD by hand

### 3.1 Step 1: $A^TA$
The detour via $A^TA$ is taken because this matrix is small (2×2), symmetric and easy to decompose. Insert $A = U\Sigma V^T$:

$$A^TA = V\Sigma U^T\,U\Sigma V^T = V\,\Sigma^2\,V^T \qquad (U^TU = I)$$

This is an **eigendecomposition**: the eigenvalues of $A^TA$ are the **squared singular values** $\lambda_i = s_i^2$, and its eigenvectors are the columns of $V$. You already have $A^TA$ from §1.1: `N` = [[5, 10], [10, 30]], the same matrix as in the normal equations.

### 3.2 Step 2: eigenvalues → singular values
For a symmetric 2×2 matrix $\begin{bmatrix} p & q \\ q & r \end{bmatrix}$ the eigenvalues solve $\det(A^TA - \lambda I) = 0$:

$$(p - \lambda)(r - \lambda) - q^2 = 0 \quad\Rightarrow\quad \lambda^2 - (p + r)\,\lambda + (pr - q^2) = 0$$

Here: $\lambda^2 - 35\lambda + 50 = 0$, so $\lambda_{1,2} = \big(35 \pm \sqrt{35^2 - 4 \cdot 50}\big)/2$.

**Exercise:** compute both eigenvalues `lam` with this formula (largest first) and the singular values `s` $= \sqrt{\lambda}$.

In [ ]:
p_, q_, r_ = N[0, 0], N[0, 1], N[1, 1]
# YOUR CODE HERE
print("eigenvalues lambda =", lam)
print("singular values s  =", s)
assert np.allclose(s, np.linalg.svd(A, compute_uv=False)), "compare with np.linalg.svd(A, compute_uv=False)"

### 3.3 Step 3: eigenvectors → $V$
For each eigenvalue solve $(A^TA - \lambda I)\,\mathbf{v} = \mathbf{0}$. The first row reads $(p - \lambda)\,v_a + q\,v_b = 0$, so

$$\mathbf{v} \propto (q,\ \lambda - p) \qquad \text{here } (10,\ \lambda - 5)$$

and then normalise to length 1. For $\lambda_1 = 33.508$: $(10, 28.508)$, length 30.211, $\mathbf{v}_1 = (0.331, 0.944)$.

**Exercise:** build `V` with the normalised eigenvectors as **columns** and check that $V^TV = I$.

In [ ]:
# YOUR CODE HERE
print("V =\n", V)
print("V^T V =\n", V.T @ V)
assert np.allclose(V.T @ V, np.eye(2)), "the columns of V must be orthonormal"
assert np.allclose(N @ V, V * lam), "columns of V must be eigenvectors of A^T A"

**What $V$ tells us.** $\mathbf{v}_1 = (0.331, 0.944)$ points mostly along $b$, the slope. This combination of the parameters is the one the data determine **best** (large $s_1$): a small change of the slope moves the line a lot at the ends. $\mathbf{v}_2 = (0.944, -0.331)$ points mostly along $a$, the intercept, with a small opposite change of $b$. It is determined **less well** (small $s_2$): raise the intercept and lower the slope a little, and the line tilts about a point near the middle of the data, so it hardly moves where the points are. The ratio $s_1/s_2$ will reappear as the condition number in §5.

### 3.4 Step 4: $U$ from $A\,\mathbf{v}_i = s_i\,\mathbf{u}_i$
Now $U$ follows directly:

$$\mathbf{u}_i = \frac{A\,\mathbf{v}_i}{s_i}$$

Read it as: evaluate the line with the parameters $\mathbf{v}_i$ at the five $x$ values and divide by $s_i$. For $\mathbf{v}_1$: $0.331 + 0.944\,x$ = 0.331, 1.275, 2.219, 3.163, 4.107; divided by 5.789: 0.057, 0.220, 0.383, 0.546, 0.709.

**Exercise:** compute the 5×2 matrix `U` in one line. (Hint: `A @ V / s` divides each column by its own singular value, thanks to broadcasting.)

In [ ]:
# YOUR CODE HERE
print("U =\n", U)
assert U.shape == (5, 2), "U must be 5 x 2"

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(x, U[:, 0], "o-", color="tab:blue", label="u1 = A v1 / s1")
ax.plot(x, U[:, 1], "o-", color="tab:orange", label="u2 = A v2 / s2")
ax.axhline(0, color="k", lw=0.5)
ax.set(xlabel="x", ylabel="component", title="The columns of U are lines")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

Every column of $U$ is a **line evaluated at the $x$ values**, normalised to length 1: $\mathbf{u}_1$ rises, $\mathbf{u}_2$ falls and crosses zero near $x = 3$. Every data vector that a line can explain *exactly* is a combination of $\mathbf{u}_1$ and $\mathbf{u}_2$; together they span the column space of $A$.

### 3.5 Check: $U$ is orthonormal, and $U\Sigma V^T = A$
The columns of $U$ are orthonormal by construction, not by luck:

$$\mathbf{u}_i^T\mathbf{u}_j = \frac{\mathbf{v}_i^T A^TA\,\mathbf{v}_j}{s_i s_j} = \frac{\lambda_j\,\mathbf{v}_i^T\mathbf{v}_j}{s_i s_j} = \begin{cases} 1 & i = j \\ 0 & i \ne j \end{cases}$$

because $\mathbf{v}_j$ is an eigenvector ($A^TA\,\mathbf{v}_j = \lambda_j\mathbf{v}_j$), the $\mathbf{v}$ are orthonormal and $\lambda_i = s_i^2$.

The cell below checks this, rebuilds $A$ from your three factors and compares with NumPy. NumPy may return some columns of $U$ and $V$ with the **opposite sign**: eigenvectors are only defined up to sign. That is harmless as long as $\mathbf{u}_i$ and $\mathbf{v}_i$ flip *together*, so we compare absolute values.

In [ ]:
print("U^T U =\n", U.T @ U)
print("U S V^T == A:", np.allclose(U @ np.diag(s) @ V.T, A))
print("same as NumPy up to sign:", np.allclose(np.abs(U), np.abs(U_np)), np.allclose(np.abs(V), np.abs(Vt_np.T)))

### 3.6 Why not take $U$ from $AA^T$?
By symmetry, $AA^T = U\Sigma^2U^T$, so $U$ could also come from the eigenvectors of the 5×5 matrix $AA^T$. Two reasons against it:

* The matrix is bigger: 5×5 instead of 2×2 (for GNSS with ten satellites 10×10 instead of 4×4). It has three extra eigenvalues that are zero.
* Eigenvectors are only defined up to sign. If $U$ and $V$ are computed **separately**, nothing guarantees that the signs match, and then $U\Sigma V^T \ne A$.

The cell flips the sign of $\mathbf{u}_2$. The flipped vector is still a perfectly good eigenvector of $AA^T$, yet the product no longer gives $A$. With $\mathbf{u}_i = A\mathbf{v}_i/s_i$ the sign is always consistent.

In [ ]:
print("eigenvalues of A A^T:", np.linalg.eigvalsh(A @ A.T)[::-1].round(3))

U_flip = U.copy()
U_flip[:, 1] *= -1
print("u2 flipped is still an eigenvector of A A^T:",
      np.allclose(A @ A.T @ U_flip[:, 1], lam[1] * U_flip[:, 1]))
print("but U_flip S V^T == A:", np.allclose(U_flip @ np.diag(s) @ V.T, A))

### 3.7 The recipe, and what NumPy really does
1. Form $A^TA$.
2. Compute its eigenvalues, sort them in descending order; $s_i = \sqrt{\lambda_i}$.
3. Compute the eigenvectors, normalise them: $V$.
4. $\mathbf{u}_i = A\mathbf{v}_i / s_i$: $U$.

This is the best route for **understanding**, but not for computing. Forming $A^TA$ **squares the condition number** (here from 4.7 to 22.5), and for badly conditioned problems digits get lost (§5.1). Libraries (LAPACK, called by NumPy) therefore work on $A$ directly: Householder reflections bring $A$ to bidiagonal form, then a variant of the QR iteration (Golub–Kahan) finds the singular values. The result is the same, only more stable. This is why `np.linalg.lstsq`, which uses the SVD, is preferred over `np.linalg.solve(A.T @ A, A.T @ y)`.

## 4. Least squares with the SVD

Insert $A = U\Sigma V^T$ into the least-squares problem. Split $\mathbf{y}$ into the part inside the column space, $UU^T\mathbf{y}$, and the rest $\mathbf{y}_\perp$, which no line can reach. With $\mathbf{c} = U^T\mathbf{y}$ and $\mathbf{z} = V^T\mathbf{p}$:

$$\lVert \mathbf{y} - U\Sigma V^T\mathbf{p} \rVert^2 = \lVert \mathbf{c} - \Sigma\,\mathbf{z} \rVert^2 + \lVert \mathbf{y}_\perp \rVert^2$$

The second term does not depend on $\mathbf{p}$. The first term is zero for $z_i = c_i / s_i$. So

$$\mathbf{p} = V\,\Sigma^{-1}\,U^T\mathbf{y}$$

with no system of equations to solve, only three simple steps:

| step | computation | meaning |
|---|---|---|
| 1 | $\mathbf{c} = U^T\mathbf{y}$ | project the measurements onto the directions of $U$ |
| 2 | $\mathbf{z} = \mathbf{c} / \mathbf{s}$ | undo the stretching |
| 3 | $\mathbf{p} = V\mathbf{z}$ | rotate back to the parameters $a, b$ |

$V\Sigma^{-1}U^T$ is the **pseudo-inverse** $A^+$ of $A$ (`np.linalg.pinv`).

**Exercise:** compute `c`, `z` and `p_svd` with your `U`, `s`, `V`.

In [ ]:
# YOUR CODE HERE
print("c =", c, "  z =", z, "  p =", p_svd)
assert np.allclose(p_svd, p_normal), "the SVD solution must equal the normal-equation solution"

In [ ]:
print("normal equations :", p_normal)
print("SVD by hand      :", p_svd)
print("np.linalg.lstsq  :", np.linalg.lstsq(A, y, rcond=None)[0])
print("np.linalg.pinv   :", np.linalg.pinv(A) @ y)

### 4.1 Where the residuals live
`np.linalg.svd(A)` without `full_matrices=False` returns a 5×5 $U$. Its three extra columns $\mathbf{u}_3, \mathbf{u}_4, \mathbf{u}_5$ complete $\mathbf{u}_1, \mathbf{u}_2$ to an orthonormal basis of the whole measurement space. They have no singular value: $A$ never reaches these directions. That is exactly where the **residuals** are, the part of $\mathbf{y}$ that no line can explain:

$$\mathbf{v} = \mathbf{y}_\perp = U_\perp U_\perp^T\mathbf{y}, \qquad \lVert \mathbf{v} \rVert^2 = \lVert \mathbf{y} \rVert^2 - \lVert \mathbf{c} \rVert^2$$

and, as a consequence, the residuals are perpendicular to every column of $A$: $A^T\mathbf{v} = \mathbf{0}$, which is the normal equations once more. The solution itself only needs the reduced SVD.

In [ ]:
U_full = np.linalg.svd(A)[0]
U_perp = U_full[:, 2:]                          # 5 x 3: directions no line can reach
v = y - A @ p_svd
print("residuals             :", v.round(3))
print("U_perp U_perp^T y     :", (U_perp @ U_perp.T @ y).round(3))
print("|v|^2 = |y|^2 - |c|^2 :", round(float(v @ v), 4), "=", round(float(y @ y - c @ c), 4))
print("A^T v (should be 0)   :", A.T @ v)

## 5. Stretch: conditioning

Step 2 of §4 divides by the singular values. If one of them is small, everything in that direction, **noise included**, is blown up. The ratio

$$\kappa(A) = s_{\max} / s_{\min}$$

is the **condition number**. For our five points $\kappa = 4.7$: well conditioned.

**Predict first:** use the same spacing of points, but at $x = 1000 \ldots 1004$. The points still define a line just as well. What happens to $\kappa$, and to the uncertainty of $a$ and of $b$?

The cell below fits 2000 noisy versions of the line $y = 1 + 2x$ (noise $\sigma = 0.2$) for both sets and reports the spread of the estimates.

In [ ]:
def fit_spread(x, sigma=0.2, runs=2000, seed=1):
    # Fit y = 1 + 2x + noise many times; return condition numbers and the std of a and b.
    rng = np.random.default_rng(seed)
    A_ = np.c_[np.ones_like(x), x]
    P = np.array([np.linalg.lstsq(A_, 1 + 2 * x + rng.normal(0, sigma, len(x)), rcond=None)[0]
                  for _ in range(runs)])
    s_ = np.linalg.svd(A_, compute_uv=False)
    return {"cond(A)": s_[0] / s_[-1], "cond(A^T A)": (s_[0] / s_[-1]) ** 2,
            "std a": P[:, 0].std(), "std b": P[:, 1].std()}


for name, xs in [("x = 0 ... 4", np.arange(5.0)), ("x = 1000 ... 1004", 1000 + np.arange(5.0))]:
    print(f"{name:18s}", {k: float(f"{val:.3g}") for k, val in fit_spread(xs).items()})

**Questions:** Why is $b$ as good as before, while $a$ is about 400 times worse? What is $a$ here, geometrically, and where is it measured? Try centring the points: use $x - \bar{x}$ (i.e. $-2 \ldots 2$) and look at $\kappa$ and at $V$.

In [ ]:
# YOUR CODE HERE


✍️ *Your answer:*

### 5.1 Why `lstsq` and not the normal equations
The normal equations work with $A^TA$, whose condition number is $\kappa^2$. With 16 significant digits in double precision, roughly $\log_{10}\kappa^2$ digits are lost. The cell fits **exact** points on $y = 1 + 2x$ (no noise, so the answer must be $a = 1$, $b = 2$) at growing distance from $x = 0$.

In [ ]:
for x0 in [1e3, 1e5, 1e6]:
    xs = x0 + np.arange(5.0)
    A_, y_ = np.c_[np.ones(5), xs], 1 + 2 * xs
    p_ne = np.linalg.solve(A_.T @ A_, A_.T @ y_)
    p_ls = np.linalg.lstsq(A_, y_, rcond=None)[0]
    print(f"x0 = {x0:8.0e}  cond(A) = {np.linalg.cond(A_):8.1e}  cond(A^T A) = {np.linalg.cond(A_.T @ A_):8.1e}"
          f"  normal eq.: a = {p_ne[0]:9.4f}   lstsq: a = {p_ls[0]:.4f}")

At $x \approx 10^6$ the normal equations give an intercept that is wrong by tens of units, while `lstsq` (SVD) is still correct to a few decimals. GNSS coordinates are of the order of $10^7$ m, so the solver in geo0264 uses `lstsq` on purpose (and solves for small corrections $\Delta\mathbf{p}$, which also helps).

## 6. Stretch: rank deficiency

What if all five points have the **same** $x$, say $x = 2$? Then every line through the point $(2, \bar{y})$ fits equally well: $a + 2b = \bar{y}$ has infinitely many solutions. In the SVD this shows up as a singular value of **zero**: the direction $\mathbf{v}_2 \propto (2, -1)$ (raise $a$ by 2, lower $b$ by 1) does not change $A\mathbf{p}$ at all.

`lstsq` handles this: it treats singular values below `rcond` × $s_{\max}$ as zero, skips the division by them and returns the **minimum-norm** solution (the shortest $\mathbf{p}$ among all equally good ones). It also reports the rank.

In [ ]:
x_bad = np.full(5, 2.0)
A_bad = np.c_[np.ones(5), x_bad]
print("singular values:", np.linalg.svd(A_bad, compute_uv=False))
p_bad, res, rank, sv = np.linalg.lstsq(A_bad, y, rcond=None)
print("rank:", rank, "  minimum-norm solution a, b =", p_bad)
print("a + 2b =", p_bad @ [1, 2], "  mean of y =", y.mean())

For GNSS a zero singular value means the geometry cannot separate some unknowns at all, e.g. fewer than four satellites, or satellites whose lines of sight all lie on one cone. Then $(A^TA)^{-1}$ does not exist and PDOP is infinite. Nearly zero means nearly degenerate: a huge PDOP.

## 7. Back to GNSS: covariance and PDOP from the SVD

If every observation has the same independent noise $\sigma$, the covariance of the least-squares estimate is

$$\Sigma_{\mathbf{p}} = \sigma^2 (A^TA)^{-1} = \sigma^2\, V\,\Sigma^{-2}\,V^T$$

(insert $A^TA = V\Sigma^2V^T$ from §3.1). So the SVD gives the uncertainty directly: along $\mathbf{v}_i$ the standard deviation is $\sigma / s_i$. The **weakest direction** is the $\mathbf{v}$ with the smallest $s$, and PDOP is $\sqrt{Q_{XX} + Q_{YY} + Q_{ZZ}}$ with $Q = V\Sigma^{-2}V^T$.

The cell builds the GNSS design matrix of geo0264 for six invented satellites above 10° elevation, written in local **east, north, up** instead of X, Y, Z (a rotation, which does not change PDOP). Each row is the unit vector from the satellite to the receiver plus a 1 for the clock.

In [ ]:
rng = np.random.default_rng(7)
az = np.radians(rng.uniform(0, 360, 6))
el = np.radians(rng.uniform(10, 85, 6))
los = np.c_[np.cos(el) * np.sin(az), np.cos(el) * np.cos(az), np.sin(el)]   # receiver -> satellite (E, N, U)
A_gnss = np.c_[-los, np.ones(6)]                                         # columns: E, N, U, clock

U_g, s_g, Vt_g = np.linalg.svd(A_gnss, full_matrices=False)
Q_svd = Vt_g.T @ np.diag(1 / s_g**2) @ Vt_g
Q_inv = np.linalg.inv(A_gnss.T @ A_gnss)
print("elevations [deg]:", np.degrees(el).round(0))
print("singular values :", s_g)
print("PDOP via inverse:", round(float(np.sqrt(np.trace(Q_inv[:3, :3]))), 3),
      "  via SVD:", round(float(np.sqrt(np.trace(Q_svd[:3, :3]))), 3))
print("weakest direction v (E, N, U, clock):", Vt_g[-1].round(2), " std = sigma x", round(1 / s_g[-1], 2))

**Questions:** Which components dominate the weakest direction? Why does exactly this combination come out badly determined? (Hint: all satellites are *above* the receiver. What happens to all ranges if the receiver moves up and, at the same time, the clock bias $b$ grows by about the same amount?) Change the elevation range in the cell (e.g. `uniform(40, 85, 6)`) and watch $s_{\min}$ and PDOP.

✍️ *Your answer:*

## 8. Wrap-up
1. In your own words: what are $V$, $\Sigma$ and $U$ for the line fit?
2. Why are the eigenvalues of $A^TA$ the squares of the singular values, and why can they never be negative?
3. Why does `np.linalg.lstsq` not form $A^TA$?
4. What does a small singular value mean for a line fit, and what for GNSS?

✍️ *Your answer:*

### Sources
* Wikipedia: *Singular value decomposition*, *Linear least squares*, *Moore–Penrose inverse*, *Condition number*, *Dilution of precision (navigation)*.
* G. H. Golub, C. F. Van Loan: *Matrix Computations*, 4th ed., Johns Hopkins University Press, 2013 (SVD algorithm, §8.6).
* G. Strang: *Introduction to Linear Algebra*, 5th ed., Wellesley-Cambridge Press, 2016 (chapter 7, the SVD).